## Banka Pazarlama Analizi: Vadeli Mevduat Tahminleme Modeli
#### Proje Hakkında
Bu proje, bir bankanın doğrudan pazarlama kampanyaları (telefon aramaları) sonucunda müşterilerin Vadeli Mevduat (Term Deposit) hesabı açıp açmayacağını tahmin etmek amacıyla geliştirilmiştir. Veri seti, Portekiz'deki bir bankacılık kurumuna ait gerçek kampanya verilerini içermektedir.

Projede, bankanın operasyonel verimliliğini artırmak için Logistic Regression, Random Forest, XGBoost ve SVM gibi çeşitli makine öğrenmesi algoritmaları yarıştırılmıştır.

## Değişkenlerin Tanıtımı
#### Müşteri Demografik Bilgileri
age: Müşterinin yaşı.

job: Meslek türü.

marital: Medeni durum.

education: Eğitim seviyesi.

default: Mevcut bir kredi borcu temerrüdü var mı?

housing: Konut kredisi var mı?

loan: Kişisel borç/kredi var mı?

#### Kampanya ve İletişim Detayları
contact: İletişim türü (Hücresel, telefon).

month: Son temas kurulan ay.

day_of_week: Son temas kurulan gün.

duration: Son görüşme süresi (Saniye). Not: Bu değişken tahmin yapıldıktan sonra ortaya çıktığı için analizde dikkatli değerlendirilmelidir.

campaign: Bu kampanya sırasında müşteriyle kurulan temas sayısı.

pdays: Önceki kampanyadan sonra geçen gün sayısı.

previous: Mevcut kampanyadan önce kurulan temas sayısı.

poutcome: Önceki pazarlama kampanyasının sonucu.

#### Kendi Mühendisliğim (Feature Engineering)
age_group: Yaşın segmentlere ayrılarak (Genç, Orta, Yaşlı) daha anlamlı hale getirilmesi.

is_senior_retired: Müşterinin emekli veya yaşlılık kategorisinde olup olmadığını belirten stratejik değişken.

season: Görüşme yapılan ayların mevsimsel etkilerini ölçmek için oluşturulan değişken.

pdays_contacted: Veri setindeki orijinal pdays değişkeninde bulunan "999" (iletişim kurulmadı) karmaşasını çözmek için, müşteriler "Daha önce ulaşılanlar (1)" ve "Hiç ulaşılmayanlar (0)" olarak iki net gruba ayrılmıştır. Bu dokunuş, modelin en önemli karar kriterlerinden biri olmuştur.

#### Hedef Değişken (Target)
y: Müşteri vadeli mevduat açtı mı? (yes/no).

#### Analiz Stratejisi
Veri setindeki sınıf dengesizliği (imbalance) sorunu, banka pazarlama projelerinde yaygın bir durumdur. Bu projede sadece doğruluğa (Accuracy) bakılmamış; bankanın müşteri kaçırmama gücünü ölçen Recall ve genel verimliliği ölçen F1-Score metrikleri temel karar verici olarak kullanılmıştır.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)
pd.options.display.float_format = '{:,.0f}'.format

In [ ]:
data=pd.read_csv("bank.csv",sep=";")
df=data.copy()
df.head(1)

## Data Cleaning

In [ ]:
df = df[df["age"] >= 18]

In [ ]:
df = df.reset_index(drop=True) 
# Nedeni: Yaş filtresinden sonra silinen satırların yerini doldurur,
# böylece en sonda gördüğün o "y'de NaN var mı? -> True" hatasını kökten çözer.

In [ ]:
# 999 demek "daha önce aranmadı" demek. Bunu kategorize edelim.
df['pdays_contacted'] = df['pdays'].apply(lambda x: 0 if x == 999 else 1)

# Hedef değişkeni (y) numerik yapalım
df['y'] = df['y'].map({'yes': 1, 'no': 0})

# Gereksiz veya sızıntı (data leakage) yapabilecek sütunları çıkaralım
# 'duration' sütununu çıkarıyoruz çünkü arama bitmeden süreyi bilemeyiz, tahmin modelinde olmaz.
features = [
    'age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
    'contact', 'month', 'day_of_week', 'campaign', 'pdays_contacted', 
    'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 
    'cons.conf.idx', 'euribor3m', 'nr.employed', 'y'
]
df = df[features]

In [ ]:
df.isnull().sum()

In [ ]:
df.head(1)

In [ ]:
# 1. Aykırı Değer (Outlier) Baskılama Özeti
print("--- VERİ TEMİZLEME VE BASKILAMA ÖZETİ ---")
outlier_cols = ['age', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

for col in outlier_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low_limit = q1 - 1.5 * iqr
    up_limit = q3 + 1.5 * iqr
    
    # Kaç veri bu sınırın dışındaydı?
    count = data[(data[col] < low_limit) | (data[col] > up_limit)].shape[0]
    
    if count > 0:
        print(f" {col.upper()}: {count} adet uç değer sınırlandırılarak baskılandı.")
    else:
        print(f" {col.upper()}: Aykırı değer saptanmadı, orijinal veriler korundu.")

# 2. Logaritmik Dönüşüm Özeti
print("\n--- NORMALİZASYON ÖZETİ ---")
print(f" CAMPAIGN: 1-{data['campaign'].max()} arası olan aralık, logaritmik olarak {df['campaign'].min():.2f}-{df['campaign'].max():.2f} arasına çekildi.")

Outlier Baskılama (IQR): age (469) ve cons.conf.idx (447) sütunlarındaki uç değerler baskılanarak veri stabilize edildi.

Logaritmik Dönüşüm: campaign sütununa sağa çarpık dağılımı dengelemek için np.log1p uygulandı.
Ekonomik Veriler: euribor3m ve emp.var.rate gibi sütunlarda aykırı değer saptanmadığı için orijinal değerler korundu.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

for col in outlier_cols:
    # Fonksiyon çağırmak yerine doğrudan hesaplayıp baskılıyoruz:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low_limit = q1 - 1.5 * iqr
    up_limit = q3 + 1.5 * iqr
    
    # Veriyi sınırlar arasında kırpıyoruz (Kalıcı çözüm!)
    df[col] = df[col].clip(lower=low_limit, upper=up_limit)

# Campaign log dönüşümü
if df['campaign'].max() > 10:
    df['campaign'] = np.log1p(df['campaign'])

# --- 2. ADIM: GÖRSELLEŞTİR (ÖNCESİ VS SONRASI) ---
sns.set_style('whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# AGE KARŞILAŞTIRMASI
sns.boxplot(x=data['age'], ax=axes[0, 0], color='lightgray')
axes[0, 0].set_title('Age: İşlem ÖNCESİ (Ham Veri)')

sns.boxplot(x=df['age'], ax=axes[0, 1], color='skyblue')
axes[0, 1].set_title('Age: İşlem SONRASI (Baskılanmış)')

# CAMPAIGN KARŞILAŞTIRMASI
sns.histplot(data['campaign'], bins=30, kde=True, ax=axes[1, 0], color='lightgray')
axes[1, 0].set_title('Campaign: Log Öncesi (Dağılım Bozuk)')

sns.histplot(df['campaign'], bins=20, kde=True, ax=axes[1, 1], color='salmon')
axes[1, 1].set_title('Campaign: Log Sonrası (Normalize Edilmiş)')

plt.tight_layout()
plt.show()

# --- 3. ADIM: SAYISAL KONTROL ---
print(f"Kontrol - Age Max: {df['age'].max()}")
print(f"Kontrol - Campaign Max: {df['campaign'].max():.2f}")

In [ ]:
# 60 yaş üstü ve mesleği 'retired' olanları 1, diğerlerini 0 yapalım
df['is_senior_retired'] = ((df['age'] >= 60) & (df['job'] == 'retired')).astype(int)

print(f"Yeni 'is_senior_retired' sütunundaki dağılım:\n{df['is_senior_retired'].value_counts()}")

In [ ]:
# Ayları mevsimlerle eşleştiren bir sözlük yapalım
# Not: Veri setindeki ay isimlerinin (may, jun, jul...) küçük harf olduğundan emin olalım.
seasons = {
    'mar': 'spring', 'apr': 'spring', 'may': 'spring',
    'jun': 'summer', 'jul': 'summer', 'aug': 'summer',
    'sep': 'autumn', 'oct': 'autumn', 'nov': 'autumn',
    'dec': 'winter'
}

# 'month' sütununu kullanarak 'season' sütununu oluşturalım
df['season'] = df['month'].map(seasons)

# Veri setinde Ocak/Şubat eksik , eşleşmediyse onları da 'winter' olarak dolduralım.
df['season'] = df['season'].fillna('winter')

print(f"Mevsim dağılımı:\n{df['season'].value_counts()}")

In [ ]:
# Sadece yeni sütunları ve ilgili oldukları yaş/meslek sütunlarını görerek sağlamasını yapalım
print(df[['age', 'job', 'is_senior_retired', 'season']].head(5))

In [ ]:
# is_senior_retired sütunu 1 olanlardan 2 örnek görelim
test_df = df[df['is_senior_retired'] == 1][['age', 'job', 'is_senior_retired', 'season']].head(2)
test_df

In [ ]:
# 1. Adım: Hedef değişkeni ayır (Onu koruma altına alıyoruz)
y = df['y']
X = df.drop('y', axis=1)

# 2. Adım: Sadece X (özellikler) içindeki kategorik sütunları seç
cat_cols = X.select_dtypes(include=['object']).columns

# 3. Adım: One-Hot Encoding uygula (Sadece X'e)
X_final = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# 4. Adım: Şimdi y ile X_final'ı tertemiz bir şekilde birleştir
df_final = pd.concat([X_final, y], axis=1)

print(f"Yeni veri seti boyutu: {df_final.shape}")
df_final.head()

In [ ]:
# 1. Önce veriyi özellikler (X) ve hedef (y) olarak ayırıyoruz
# y: Tahmin etmek istediğimiz "Kabul etti mi?" sorusunun cevabı
y = df['y'] 
X = df.drop('y', axis=1)

# 2. X içindeki kategorik sütunları 0-1 formatına çeviriyoruz
# Bu işlem y'ye dokunmadığı için NaN hatasını kökten çözer
X_final = pd.get_dummies(X, drop_first=True)

# 3. Sonuçları kontrol edelim
print("X_final (Özellikler) tablosunda NaN var mı?:", X_final.isnull().values.any())
print("y (Hedef) değişkeninde NaN var mı?:", y.isnull().values.any())

# Eğer her iki çıktı da False ise, veri iyi olmuş demektir!
X_final.head()

In [ ]:
df.describe()

In [ ]:
df.shape

## Binning (Kutulama)

In [ ]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[18, 30, 45, 60, 100],
    labels=["genc", "orta", "ileri_orta", "yasli"]
)

## EDA(KEŞİFSEL VERİ ANALİZİ)

### Yaş Gruplarına Göre Kampanya Kabul Oranı

In [ ]:
result = (
    df.groupby("age_group")["y"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("ratio")
    .reset_index()
)

yes_rates = result[result["y"] == 1]

plt.figure(figsize=(7,5))

plt.bar(
    yes_rates["age_group"],
    yes_rates["ratio"]
)

plt.xlabel("Yaş Grubu")
plt.ylabel("Kabul Oranı (%)")
plt.title("Yaş Gruplarına Göre Kampanya Kabul Oranı")

plt.show()

## Yaş ve Medeni Durum (Marital) Çaprazlaması
Bir müşteri sadece "yaşlı" olduğu için mi mevduat açıyor, yoksa "evli/bekar" olması bu durumu değiştiriyor mu? Örneğin, "Genç ve bekar" olanlar mı daha potansiyelli, yoksa "Genç ve evli" olanlar mı?

In [ ]:
print("age_group sütunu var mı?:", "age_group" in df.columns)

In [ ]:
# Yaş grubu ve Medeni Duruma göre kabul oranları
plt.figure(figsize=(12, 6))
sns.barplot(x='age_group', y='y', hue='marital', data=df, palette='muted')
plt.title('Yaş Grubu ve Medeni Duruma Göre Kabul Oranı')
plt.show()

banka eğer reklam bütçesini verimli kullanmak istiyorsa, yaşlı müşterilere (özellikle evli ve bekar ayrımı yapmaksızın) ve genç bekarlara odaklanmalı. Orta yaşlı evli kitleye yapılacak aramalar muhtemelen zaman kaybı olacaktır.

## Yaş ve Eğitim (Education) İlişkisi
Eğitim seviyesi arttıkça, belirli yaş gruplarının bankacılık ürünlerine olan güveni veya ilgisi artıyor mu? Bu grafik, bankanın reklam dilini kime göre (akademik mi, daha sade mi) seçmesi gerektiğini söyler.


In [ ]:
# Yaş grubu ve Eğitim seviyesi
plt.figure(figsize=(14, 7))
sns.barplot(x='age_group', y='y', hue='education', data=df)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('Yaş Grubu ve Eğitime Göre Kabul Oranı')
plt.show()

In [ ]:
# Eğitim gruplarında kaçar kişi olduğunu görelim
print(df['education'].value_counts())

# Okuma yazma bilmeyenlerin kaçı kabul etmiş?
print("\nOkuma yazma bilmeyenlerin detaylı sayıları:")
print(df[df['education'] == 'illiterate']['y'].value_counts())

İstatistiksel olarak 18 kişilik bir kitle, 41.000 kişilik bir veri setinde genelleme yapmak için çok küçüktür. Bu yüzden biz bu tür "uç" başarıları raporlarken "Örneklem sayısı yetersiz olduğundan bu veri yanıltıcı olabilir" şerhini düşeriz.

### Yaşlılar arasında çalışanlar mı yoksa emekliler mi daha çok kabul ediyor?
Yaşlılar zengin oldukları için mi (işlerine/mesleklerine göre) kabul ediyorlar, yoksa sadece yaşlılık psikolojisi mi?

In [ ]:
plt.figure(figsize=(15, 8))

# Yaş grupları ve Mesleklerin kabul oranı üzerindeki ortak etkisi
sns.barplot(x='age_group', y='y', hue='job', data=df, palette='tab20')

plt.title('Yaş Grupları ve Mesleklere Göre Kampanya Kabul Oranı', fontsize=15)
plt.ylabel('Kabul Oranı (y)')
plt.xlabel('Yaş Grubu')
plt.legend(title='Meslekler', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

Bu tablo bize şunu söylüyor: Yaş, meslekten daha baskın bir faktördür. 
1.  Yaşlılık Etkisi: 60 yaşın üzerindeki müşterilerde meslek ne olursa olsun (ev hanımından yöneticiye kadar) kabul oranları %40'ın üzerine çıkıyor. Bu, yaşlı bireylerin riskten kaçınma ve birikimlerini güvenli liman olan mevduatta tutma eğiliminin evrensel olduğunu gösterir.

2.  Orta Yaş Çıkmazı: 30-60 yaş arası kitle (orta ve ileri_orta), bankanın en zorlandığı kitle. Bu gruptaki insanların iş unvanları ne kadar yüksek olursa olsun (Management, Admin), kabul oranları hep düşük kalmış. Çünkü bu yaş grubu "borç ödeme ve harcama" dönemindedir.

3.  Gençlerin Potansiyeli: Gençlerde özellikle "öğrenci" ve "idari personel" (admin) grupları, erken yaşta tasarrufa başlamaya daha açık görünüyor.

### Ekonomik Dalgalanmalar ve Başarı 
Müşterilerin mevduat açma kararı piyasadaki faiz oranlarına (euribor3m) ne kadar bağlı? Burada çubuk grafik yerine KDE Plot kullanarak, başarının hangi faiz aralıklarında "kümelendiğini" görelim.

In [ ]:
plt.figure(figsize=(10, 6))
# Başarılı (1) ve Başarısız (0) aramaların faiz oranına göre dağılımı
sns.kdeplot(data=df, 
            x='euribor3m', 
            hue='y', 
            fill=True, 
            common_norm=False, 
            palette='Set1', 
            alpha=0.5)

plt.title('Faiz Oranları (Euribor 3M) Başarıyı Nasıl Etkiliyor?', fontsize=14)
plt.xlabel('Faiz Oranı')
plt.ylabel('Yoğunluk (Arama Sıklığı)')
plt.show()

Biz en çok aramayı faizler 5 iken yapmışız ama en çok kabulü faizler 1 iken almışız. Gelecek kampanyayı faizlerin düştüğü dönemlere saklarsak başarı oranımız 3-4 kat artacaktır.

### Tüketici Güveni ve Ekonomi 
İnsanlar ekonomiye güven duyduklarında mı (cons.conf.idx) yoksa fiyatlar arttığında mı (cons.price.idx) paralarını bankaya yatırıyorlar?

In [ ]:
plt.figure(figsize=(10, 6))
# Tüketici Güveni vs Tüketici Fiyat Endeksi
# 's' parametresi noktaların boyutunu, 'alpha' şeffaflığını ayarlar
sns.scatterplot(data=df, x='cons.conf.idx', y='cons.price.idx', hue='y', alpha=0.9, palette='coolwarm')

plt.title('Ekonomik Güven ve Fiyat Endeksi İlişkisi', fontsize=14)
plt.xlabel('Tüketici Güven Endeksi')
plt.ylabel('Tüketici Fiyat Endeksi')
plt.legend(title='Kabul Durumu (y)', bbox_to_anchor=(1.05, 1))
plt.show()

Ekonomiye güvenin çok düşük olduğu dönemlerde kampanya kabul oranı düşmektedir.Buna karşılık enflasyonun hissedildiği ve belirsizliğin orta seviyede olduğu dönemlerde müşteriler mevduat tekliflerine daha fazla olumlu dönüş yapmıştır. Mavi noktaların yoğun olduğu bölgelerde insanlar ekonomiye hiç güvenmiyor.

### Mevsimsel "Isı" Haritası
Şimdi de zamanlamaya bakalım. Mevsimleri oluşturduk ama hangi mevsimde, hangi ekonomik şartlar (örneğin faizler) altında daha başarılıyız? Bunu bir Isı Haritası (Heatmap) ile görelim.

In [ ]:
# Mevsim ve başarı oranını içeren özet tablo
heatmap_data = df.pivot_table(index='season', values='y', aggfunc='mean')

plt.figure(figsize=(8, 5))

sns.heatmap(heatmap_data, annot=True, cmap='YlGnBu', 
            cbar_kws={'label': 'Başarı Oranı (Kabul %)'})

plt.title('Mevsimlere Göre Başarı Yoğunluğu', fontsize=14)
plt.ylabel('Mevsim')
plt.show()

 Kampanya analizine göre ilkbahar ve yaz aylarında müşteri kabul oranları daha düşüktür.
 Bu nedenle bütçe ve personel yoğunluğunun Eylül-Aralık dönemine kaydırılması daha verimli olabilir.
 Çünkü kış döneminde bir müşteriyi ikna etme olasılığı yaz dönemine göre yaklaşık 4 kat daha fazladır.

### Hangi ay daha başarılı?
verideki "makro" bakış açısını "mikro" detayla destekleyerek daha derinden bakalım.

In [ ]:
# 1. Oranları hesaplayalım
month_rates = (
    df.groupby("month")["y"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)

# 2. Sadece "yes" (1) olanları alalım
month_yes = month_rates[month_rates["y"] == 1].copy()

# 3. Ayları doğru takvim sırasına sokalım (Sıralama hatasını çözen kısım)
month_order = ['mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
month_yes['month'] = pd.Categorical(month_yes['month'], categories=month_order, ordered=True)
month_yes = month_yes.sort_values('month')

# 4. Görselleştirme
plt.figure(figsize=(10,5))
plt.bar(month_yes["month"], month_yes["proportion"], color='skyblue', edgecolor='black')

plt.xlabel("Ay")
plt.ylabel("Kabul Oranı (%)")
plt.title("Aylara Göre Kampanya Kabul Oranı (Takvim Sıralı)")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

### Müşteriyi iknanın Süresi (duration)
Müşteriyle ne kadar konuştuğumuzun kabul oranına etkisini görelim. 

In [ ]:
plt.figure(figsize=(10, 6))
# Veriyi 1500 saniye ile sınırlayalım ki grafik ezilmesin (Çoğu arama bu aralıkta)
data_filtered = data[data['duration'] < 1500]

sns.violinplot(x='y', y='duration', data=data_filtered, palette='Pastel1', inner="quartile")

plt.title('Arama Süresi Yoğunluğu (0-1500 Saniye Arası)', fontsize=14)
plt.xlabel('Kabul Durumu (0: Hayır, 1: Evet)')
plt.ylabel('Süre (Saniye)')
plt.show()

Grafik bize şunu net bir şekilde söylüyor: Sabır, satışı getirir. Müşteriyi ilk 2 dakikada ikna edememiş olsak bile, görüşme süresi uzadıkça 'Evet' alma ihtimalimiz ciddi oranda artıyor. 500 saniyeyi aşan görüşmelerde başarı oranı zirveye çıkıyor.

## Eski Dostlar: poutcome (Önceki Kampanya Sonucu)
Bu analiz, bankanın geçmişte başarılı olduğu müşterilerin sadakatini ölçer. Genellikle "Eski müşteriyi tutmak, yenisini kazanmaktan daha ucuzdur" kuralının kanıtıdır.

In [ ]:
# Gruplayıp oranları hesaplayalım
poutcome_success = df.groupby('poutcome')['y'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(9, 5))
sns.barplot(x='poutcome', y='y', data=poutcome_success, palette='viridis')

plt.title('Önceki Kampanya Sonucunun Şimdiki Başarıya Etkisi', fontsize=14)
plt.ylabel('Başarı Oranı (Kabul %)')
plt.xlabel('Önceki Kampanya Sonucu')
# Y eksenini yüzde formatına çevirelim (isteğe bağlı)
plt.gca().set_yticklabels(['{:.0f}%'.format(x*100) for x in plt.gca().get_yticks()])
plt.show()

 Önceki kampanyada başarılı dönüş yapan müşterilerin mevcut kampanyayı kabul etme oranı oldukça yüksektir.
 Buna karşılık önceki kampanyası başarısız olan veya daha önce hiç kampanyaya katılmayan müşterilerde kabul oranı düşüktür.

## Israrın Sınırı: campaign (Arama Sayısı)

In [ ]:

#  'yes' -> 1, 'no' -> 0 yapıyoruz
#  daha önce yapmadım bu satırı kategorik değişkenden sayısala çeviridik:
if data['y'].dtype == 'O': 
    data['y'] = data['y'].map({'yes': 1, 'no': 0})

plt.figure(figsize=(12, 6))

# İlk 10 arama ile sınırlayalım
campaign_data = data[data['campaign'] <= 10]
# Artık sayısal olduğu için .mean() hata vermeyecek!
campaign_stats = campaign_data.groupby('campaign')['y'].mean().reset_index()

sns.lineplot(x='campaign', y='y', data=campaign_stats, marker='o', color='red', linewidth=2.5)

plt.title('Arama Sayısı Arttıkça Başarı Oranı (Kabul %) Nasıl Değişiyor?', fontsize=14)
plt.xlabel('Bu Kampanyadaki Arama Sayısı (Israr)')
plt.ylabel('Başarı Oranı')
plt.xticks(range(1, 11))
plt.grid(True, linestyle='--', alpha=0.5)

# Yüzde değerlerini noktaların üzerine yazalım
for x, y in zip(campaign_stats['campaign'], campaign_stats['y']):
    plt.text(x, y + 0.005, '{:.1%}'.format(y), ha='center')

plt.show()

Grafiğimiz gösteriyor ki; bir müşteriyi 3 kereden fazla aramak enerjimizi boşa harcamaktır. İlk 3 aramada 'Evet' alamazsak, o müşteriyi listeden çıkarıp enerjimizi hiç aranmamış (şansı %13 olan) yeni adaylara aktarmalıyız. Böylece toplam satışımızı zahmetsizce artırabiliriz.

### Müşterilerin çoğu hangi gruplardan oluşuyor?


In [ ]:
# Mesleklerin sayısal dağılımını alalım
job_counts = data['job'].value_counts()

plt.figure(figsize=(10, 10))

# Pasta grafiğini çizelim
plt.pie(job_counts, 
        labels=job_counts.index, 
        autopct='%1.1f%%', # Yüzdeleri gösterir
        startangle=140, 
        colors=plt.cm.Paired.colors) # Renk paleti

plt.title('Banka Müşterilerinin Meslek Dağılımı', fontsize=15)
plt.show()

Elimizdeki listenin yarısı admin ve mavi yakalılardan oluşuyor ancak bu grupların ikna olma oranları düşük. Öte yandan, bizi en çok seven (en yüksek kabul oranına sahip) emekliler ve öğrenciler pastanın sadece %6.3'ünü oluşturuyor. Eğer daha başarılı bir kampanya istiyorsak, pazarlama bütçesini bu küçük ama 'sadık' dilimleri genişletmek için harcamalıyız.

### Hangi meslek grubu daha çok kabul ediyor

In [ ]:
job_rates = (
    df.groupby("job")["y"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("ratio")
    .reset_index()
)

job_yes = job_rates[job_rates["y"] == 1]

In [ ]:
plt.figure(figsize=(12,5))

plt.bar(job_yes["job"], job_yes["ratio"])

plt.xticks(rotation=45)
plt.ylabel("Kabul Oranı (%)")
plt.title("Mesleğe Göre Kampanya Kabul Oranı")

plt.show()

Öneri: Gelecek kampanyalarda reklam bütçemizi emekli derneklerine veya öğrenci platformlarına kaydırarak portföydeki bu küçük dilimleri büyütmeliyiz.

### Evli insanlar mı daha çok kabul ediyor

In [ ]:
df.groupby("marital")["y"].value_counts(normalize=True) * 100

In [ ]:
# Her gruptaki toplam kişi sayısını görelim
marital_counts = df['marital'].value_counts()
print(marital_counts)

# Hem sayıları hem de oranları bir arada görelim
marital_analysis = df.groupby('marital')['y'].agg(['count', 'mean'])
marital_analysis['mean'] = marital_analysis['mean'] * 100
print(marital_analysis)

Burada gördüğümüz tam olarak "Bilinmeyenin Aldatmacası" yani ilk çıkan sonuçta bilinmeyen yüzdesi daha fazla çıkmıştır fakat veri setinde az bi yer kaplamaktadır o yüzden dikkate almamız gereken single değişkenidir.
unknown grubundaki %15'lik oran istatistiksel bir sapmadan ibarettir; bu nedenle pazarlama stratejileri bu grup üzerine değil, verisi güçlü ve tutarlı olan Bekar kitle üzerine kurulmalıdır.

## One-Hot Encoding
Her kategoriyi ayrı bir sütuna dönüştürdük.

In [ ]:
# 1. Tüm kategorik sütunları (metin tabanlı olanları) tek bir listede toplayalım
all_categorical_cols = [
    'job', 'marital', 'education', 'default', 'housing', 'loan', 
    'contact', 'month', 'poutcome', 'day_of_week', 'season', 'age_group'
]

# 2. pd.get_dummies ile 0 ve 1'lere dönüştürelim
# drop_first=True Stratejisi: Değişkenler arasındaki mükemmel bağımlılığı (multicollinearity) önlemek için her kategoriden bir seçeneği referans olarak eledik. Eğer tüm diğer seçenekler 0 (False) ise, model o kişinin elediğimiz kategoride olduğunu zaten anlar.
df_encoded = pd.get_dummies(df, columns=all_categorical_cols, drop_first=True)

print(f"Başlangıçtaki Sütun Sayısı: {df.shape[1]}")
print(f"Encoding Sonrası Sütun Sayısı: {df_encoded.shape[1]}")

# İlk 5 satıra bakarak her şeyin sayısal (True/False veya 0/1) olduğunu doğrulayalım
df_encoded.head()

Görselleştirmede (EDA) isimleri kullandık ki hikayeyi anlayalım; modellemede (Encoding) sayıları kullanıyoruz ki bilgisayar tahmin yapabilsin.

In [ ]:
# 1. En önemli değişkenleri seçelim (Çok kalabalık olmaması için)
# Hedef değişken y ile en yüksek korelasyona sahip ilk 15 değişkeni alalım
top_15_features = df_encoded.corr()['y'].abs().sort_values(ascending=False).head(15).index
df_top = df_encoded[top_15_features]

# 2. Korelasyon matrisini hesaplayalım
corr_matrix = df_top.corr()

# 3. Gerçek Isı Haritası (Heatmap) çizimi
plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f', linewidths=0.3)

# Başlığı senin istediğin gibi güncelliyoruz
plt.title('Vadeli Mevduat Kabulünü (y) Etkileyen Faktörlerin Korelasyon Matrisi', fontsize=12)
plt.show()

Modelimiz eğitim aşamasında en çok geçmiş kampanya sonuçlarına, müşterinin yaş/emeklilik durumuna ve ekonomik verilere bakarak karar verecek.

## Makine Öğrenmesi(ML)

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Hedef değişken (y) ve bağımsız değişkenleri (X) ayıralım
X = df_encoded.drop('y', axis=1)  # y hariç her şey
y = df_encoded['y']               # Sadece y (hedef)

# random_state=42: Sonuçların her çalıştırdığında aynı çıkması için (YBS projelerinde standarttır)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Eğitim seti boyutu: {X_train.shape}")
print(f"Test seti boyutu: {X_test.shape}")

#### Logistic Regression (Lojistik Regresyon)
Bu model, veri biliminde "temel taş" kabul edilir. Karmaşık modellere geçmeden önce verinin ne kadar "doğrusal" bir ilişkiye sahip olduğunu anlamamızı sağlar.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# 1. Modeli tanımlayalım (max_iter=2000: Veri setin büyük olduğu için modelin yakınsamasına zaman tanıyalım)
log_model = LogisticRegression(max_iter=2000)

# 2. Modeli eğitim verisiyle eğitelim
log_model.fit(X_train, y_train)

# 3. Test seti üzerinde tahmin yapalım
y_pred_log = log_model.predict(X_test)

# 4. Sonuçları değerlendirelim
print("--- Logistic Regression Sonuçları ---")
print(f"Doğruluk Skoru (Accuracy): {accuracy_score(y_test, y_pred_log):.4f}")
print("\nSınıflandırma Raporu:\n", classification_report(y_test, y_pred_log))

 Modelde overfittingten çok sınıf dengesizliği (imbalanced data) problemi görülmektedir.
 Model, veri setindeki çoğunluk sınıfı olan "Hayır (0)" tahminlerine yönelmiştir.
 Bu nedenle accuracy yüksek görünse de "Evet (1)" sınıfındaki müşterilerin büyük kısmı kaçırılmaktadır.
 Özellikle recall değerinin düşük olması, modelin potansiyel müşterileri yeterince yakalayamadığını göstermektedir.

#### Random Forest (Rastgele Orman)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
# 1. Modeli tanımlayalım
# n_estimators=100: 100 farklı ağaç kurup oylama yapacak
# class_weight='balanced': "dengesizlik" sorununu çözmesi için verdiğimiz komut! yani "ceza puanı". yani Model bir hata yaptığında, bu hatanın "maliyetini" hesaplar.Azınlık sınıftaki ("Evet") bir örneği yanlış tahmin etmenin cezası, çoğunluk sınıftaki ("Hayır") bir hatadan çok daha yüksektir.
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

# 2. Modeli eğitelim
rf_model.fit(X_train, y_train)

# 3. Tahmin yapalım
y_pred_rf = rf_model.predict(X_test)

# 4. Sonuçları görelim
print("--- Random Forest Sonuçları ---")
print(f"Doğruluk Skoru (Accuracy): {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nSınıflandırma Raporu:\n", classification_report(y_test, y_pred_rf))

#### Veriyi "Zorla" Dengelemek (SMOTE)
Nasıl Çalışır? Azınlık olan "Evet" (1) sınıfından yeni ve yapay örnekler üretir.

Dengeleme: Eğitim setindeki "Evet" sayısını "Hayır" sayısına eşitleyene kadar sanal veriler oluşturur.

Modelin Bakışı: Model artık sınıfları 50/50 gördüğü için özel bir ceza puanına ihtiyaç duymaz; çünkü her iki sınıf da eşit ağırlıktadır.

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

# 1. SMOTE ile veriyi dengeleyelim (Sadece eğitim setine uyguluyoruz!)
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Yeni Eğitim Seti Dengesi: \n{y_train_balanced.value_counts()}")

# 2. Random Forest'ı bu dengeli veriyle tekrar eğitelim
rf_balanced = RandomForestClassifier(n_estimators=100, random_state=42)
rf_balanced.fit(X_train_balanced, y_train_balanced)

# 3. Tahmin ve Sonuçlar
y_pred_smote = rf_balanced.predict(X_test)

print("--- SMOTE Sonrası Random Forest Sonuçları ---")
print(classification_report(y_test, y_pred_smote))

In [ ]:
!pip install xgboost

#### XGBoost (Dengesiz Verinin Gerçek Düşmanı)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# 1. scale_pos_weight hesaplayalım (Negatif / Pozitif oranı)
# Bu modelin 'Evet'lere ne kadar daha fazla önem vereceğini belirler.
ratio = (y_train == 0).sum() / (y_train == 1).sum()

# 2. XGBoost modelini kuralım
xgb_model = XGBClassifier(scale_pos_weight=ratio, random_state=42, use_label_encoder=False, eval_metric='logloss')

# 3. Modeli eğitelim (SMOTE'suz, orijinal dengesiz eğitim verisiyle)
xgb_model.fit(X_train, y_train)

# 4. Tahmin ve Sonuçlar
y_pred_xgb = xgb_model.predict(X_test)

print("--- XGBoost (Dengelenmiş Ağırlıklı) Sonuçları ---")
print(classification_report(y_test, y_pred_xgb))

#### Support Vector Machines (SVM)
Bu model, özellikle benim oluşturduğum age_group ve season gibi özelliklerin arasındaki ince farkları yakalayabilir.

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# 1. Modeli tanımlayalım 
# class_weight='balanced': Dengesizliği SVM'de de yönetiyoruz.
# probability=True: Modelin tahmin olasılıklarını da görebilmemizi sağlar.
svm_model = SVC(kernel='linear', class_weight='balanced', random_state=42)

# 2. Modeli eğitelim
# Not: SVM büyük veri setlerinde biraz daha yavaş çalışabilir.
svm_model.fit(X_train, y_train)

# 3. Tahmin ve Sonuçlar
y_pred_svm = svm_model.predict(X_test)

print("--- SVM (Support Vector Machines) Sonuçları ---")
print(classification_report(y_test, y_pred_svm))

#### 4 Algoritmayı Karşılaştıralım

In [ ]:
import pandas as pd

# 1. Görüntüleme Ayarı
pd.options.display.float_format = '{:.4f}'.format

# 2. Verileri Hazırlayalım nin (sonuçlara dayanarak)
comparison_data = {
    'Model': [
        'Logistic Regression', 
        'Random Forest (SMOTE)', 
        'XGBoost (Ağırlıklı)', 
        'SVM (Balanced)'
    ],
    'Accuracy': [0.8998, 0.8909, 0.8400, 0.6700],
    'Precision (İsabet Oranı)': [0.6700, 0.4900, 0.3600, 0.2200],
    'Recall (Yakalama Gücü)': [0.2200, 0.3800, 0.5800, 0.7300],
    'F1-Score (Genel Denge)': [0.3300, 0.4200, 0.4400, 0.3300],
    'Tahmin Edilen "Evet"': [303, 717, 1489, 3064],
    'Stratejik Değerlendirme': [
        'Çok Tutucu - Fırsat Kaçırıyor', 
        'Zayıf Denge - Geliştirilmeli', 
        'EN DENGELİ (Şampiyon)', 
        'Çok Agresif - Yüksek Maliyet'
    ]
}

df_final = pd.DataFrame(comparison_data)

print("--- Vadeli Mevduat Tahmin Modelleri Stratejik Karşılaştırması ---")
display(df_final)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Modellerin tahminlerini bir liste yapalım
models = [
    ('Logistic Regression', y_pred_log),
    ('Random Forest (SMOTE)', y_pred_smote),
    ('XGBoost (Şampiyon )', y_pred_xgb),
    ('SVM (Balanced)', y_pred_svm)
]

# 2. Görselleştirme için 2x2'lik bir alan oluşturalım
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Modellerin Hata Matrisi (Confusion Matrix) Karşılaştırması', fontsize=18)

axes = axes.flatten()

for i, (name, pred) in enumerate(models):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Blues', cbar=False)
    axes[i].set_title(name, fontsize=14)
    axes[i].set_xlabel('Tahmin Edilen')
    axes[i].set_ylabel('Gerçek Değer')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

Modellerin hata matrisleri incelendiğinde; bankanın operasyonel maliyeti ile müşteri kazanma verimliliği arasındaki en ideal dengenin XGBoost modelinde olduğu görülmüştür. Bu model, baz modele göre müşteri yakalama başarısını yaklaşık 2.7 kat artırırken, yanlış alarm oranını operasyonel olarak karşılanabilir bir düzeyde tutmaktadır.

### Feature Importance (Özellik Önemi)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. XGBoost modelinden özellik önemlerini alalım
# 'gain' metriği, bir özelliğin modelin doğruluğuna ne kadar katkı sağladığını gösterir
importance = xgb_model.get_booster().get_score(importance_type='gain')

# 2. Verileri düzenleyelim
importance_df = pd.DataFrame({
    'Feature': list(importance.keys()),
    'Importance': list(importance.values())
}).sort_values(by='Importance', ascending=False)

# 3. Görselleştirme (En önemli 15 özellik)
plt.figure(figsize=(12, 8))
top_features = importance_df.head(15)

# Oluşturduğum yeni değişkenleri vurgulamak için renk paleti ayarlayalım
colors = ['orange' if 'age' in f or 'senior' in f else 'teal' for f in top_features['Feature']]

plt.barh(top_features['Feature'], top_features['Importance'], color=colors)
plt.gca().invert_yaxis()  # En önemli olanı en üste al
plt.title('XGBoost Modelinde Karar Verici En Önemli 15 Faktör', fontsize=16)
plt.xlabel('Önem Skoru (Gain)', fontsize=12)
plt.ylabel('Değişkenler', fontsize=12)

# Değişkenlerin yanına değerlerini yazalım
for index, value in enumerate(top_features['Importance']):
    plt.text(value, index, f' {value:.2f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

 XGBoost modelinde en önemli değişken nr.employed (çalışan kişi sayısı göstergesi) olmuştur.
 Bu durum ekonomik istihdam seviyesinin müşterilerin yatırım kararlarında önemli etkisi olduğunu göstermektedir.

 month_oct (Ekim ayı) değişkeninin üst sıralarda yer alması,
 Ekim ayında yapılan kampanyaların daha başarılı olabileceğini göstermektedir.

 Feature engineering ile oluşturulan age_group_orta (orta yaş grubu) değişkeninin önemli faktörler arasına girmesi,
 yaş segmentasyonunun model tarafından anlamlı bulunduğunu göstermektedir.

 Ayrıca pdays_contacted (müşteriyle daha önce iletişime geçilip geçilmediği),
 euribor3m (3 aylık Euribor faiz oranı) ve
 cons.conf.idx (tüketici güven endeksi) gibi değişkenlerin öne çıkması,
 müşteri davranışlarının yalnızca banka içi verilerden değil,
 ekonomik koşullardan da etkilendiğini göstermektedir.

 
# Banka Pazarlama Modeli: Stratejik Yönetici Özeti

### 1. Projenin Amacı ve Kapsamı

Bu çalışma, bankanın vadeli mevduat kampanyalarına (Term Deposit) olumlu yanıt verecek potansiyel müşterileri önceden tahmin ederek, pazarlama bütçesini ve çağrı merkezi verimliliğini optimize etmek amacıyla gerçekleştirilmiştir.

### 2. Metodoloji ve Veri Hazırlığı

* **Feature Engineering:** Standart banka verilerine ek olarak, müşteri segmentasyonunu güçlendirmek amacıyla `is_senior_retired` (Emekli/Yaşlı) ve `age_group` (Yaş Grubu) gibi özgün değişkenler üretilmiştir.
* **Sınıf Dengesizliği (Imbalance):** Mevduat açan müşterilerin azınlıkta olması nedeniyle **SMOTE** (Sentetik Örnekleme) ve algoritmik ağırlıklandırma yöntemleri kullanılarak modelin "Evet" diyenleri ıskalaması engellenmiştir.

### 3. Model Performans Karşılaştırması

Dört farklı algoritma (Logistic Regression, Random Forest, XGBoost, SVM) birbiriyle yarıştırılmış ve şu sonuçlar elde edilmiştir:

| Kriter | Logistic Regression | Random Forest | **XGBoost (Şampiyon)** | SVM |
| --- | --- | --- | --- | --- |
| **Doğruluk (Accuracy)** | %90 (Yanılmaya meyilli) | %89 | **%84** | %67 |
| **Müşteri Yakalama (Recall)** | %22 (Çok düşük) | %38 | **%58 (İdeal)** | %73 (Maliyetli) |
| **Operasyonel Verim** | Düşük | Orta | **Yüksek** | Kritik |

### 4. Neden XGBoost?

Yapılan analizler sonucunda **XGBoost**, banka için en kârlı model seçilmiştir. Çünkü:

* Müşteri yakalama gücünü baz modele göre **2.7 kat** artırmıştır.
* Çağrı merkezine gereksiz arama yaptırma (Yanlış Alarm) oranını, SVM modeline göre çok daha düşük ve yönetilebilir bir seviyede tutmaktadır.

---

##  Stratejik Öneriler ve Aksiyon Planı

###  Segmentasyon ve Hedefleme

* **Yaş Grubu Etkisi:** Model kararlarında `age_group_yasli` ve `is_senior_retired` değişkenleri önemli rol oynamaktadır. Pazarlama faaliyetlerinde emekli ve orta-üst yaş grubuna yönelik özel faiz oranları sunulmalıdır.
* **Ekonomik Zamanlama:** Model, mevduat eğiliminin istihdam oranları (`nr.employed`) ve Ekim ayı (`month_oct`) ile doğrudan ilişkili olduğunu göstermiştir. Kampanyaların genel ekonomik güven endeksinin yüksek olduğu dönemlerde ve yılın son çeyreğinde yoğunlaştırılması önerilir.

###  Çağrı Merkezi Optimizasyonu

* Model tarafından üretilen **1.489 kişilik öncelikli liste** üzerinden aramalar yapılmalıdır. Bu strateji, rastgele arama yapmaya kıyasla operasyonel maliyeti düşürürken, geri dönüş oranını (Conversion Rate) maksimize edecektir.

###  Gelecek Çalışmalar

* Görüşme süresi (`duration`) modelin kararlarında etkilidir; ancak bu bilgi müşteri aranmadan önce bilinmemektedir. Gelecekte, görüşme süresine bakmaksızın tahmin yapan daha "saf" bir model üzerinde çalışılarak, kampanya öncesi tahmin doğruluğu daha da artırılabilir.

---

> "Bu modelleme süreci, veri biliminin finansal karar destek sistemlerindeki gücünü kanıtlamaktadır. Teknik başarı (Accuracy), iş hedefleriyle (Profitability) harmanlanarak banka için en optimize yol haritası çizilmiştir."